## 데이터 셋 : chatbot data
https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv

In [1]:
import torch
from datasets import load_dataset
from transformers import PreTrainedTokenizerFast, GPT2LMHeadModel

In [2]:
#공통: 토크나이저
# 한국어를 토큰화해야하므로 사전 학습된 한국어 GPT 토크나이저를 그대로 가져와서 사용

MODEL_NAME = "skt/kogpt2-base-v2"
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    MODEL_NAME,
    bos_token='</s>',
    eos_token='</s>',
    unk_token='<unk>',
    pad_token='<pad>',
    mask_token='<mask>'
)
# <usr>, <bot>을 special token으로 추가 (쪼개지지 않게)
special_tokens = {"additional_special_tokens": ["<usr>", "<bot>"]}
num_added = tokenizer.add_special_tokens(special_tokens)

vocab_size = len(tokenizer)  # 토큰 추가로 늘어난 크기 반영


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json:   0%|          | 0.00/2.83M [00:00<?, ?B/s]

In [3]:
## songys/Chatbot_data : 질문-답변 쌍 (단순 Q&A)
def load_chatbot_data(block_size=64, batch_size=32, device="cpu"):
    """
    songys/Chatbot_data를 로드해서 "<usr> 질문 <bot> 답변" 형태의 한 줄 텍스트로 합치고
    전체를 토큰화해서 하나의 긴 토큰 시퀀스로 이어 붙인다.
    데이터가 작기 때문에 GPT 사전학습처럼 전체를 이어 붙여서 학습하는 방식이 잘 맞다
    """

    # huggingface hub에 정식 등록된 형태가 아니기때문에 보통 CSV를 직접 받아서 사용
    ds = load_dataset(
        "csv",
        data_files="https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv"
    )["train"]
    # ds[i] = {"Q": "질문 텍스트", "A": "답변 텍스트", "label": 0/1/2}
    # 0=일상, 1=이별/부정, 2=사랑/긍정

    # 모든 질문-답변 쌍을 "<usr> Q <bot> A" 형태의 한 문자열로 합침
    texts = [f"<usr> {row['Q']} <bot> {row['A']}" for row in ds]
    full_text = "\n".join(texts) # 전체를 하나의 긴 텍스트로 연결

    # 토큰화: 텍스트 -> 토큰 id 리스트 (전체를 한번에 인코딩)
    all_ids = tokenizer.encode(full_text)
    data = torch.tensor(all_ids, dtype=torch.long)

    # train/val 분리 (9:1)
    n = int(len(data)* 0.9)
    train_data, val_data = data[:n], data[n:]

    def get_batch(split="train"):
        # split에 따라 train/val 데이터 중 하나를 선택
        d = train_data if split == "train" else val_data
        #배치 크기만큼 랜덤한 시작 위치를 뽑음
        ix = torch.randint(0, len(d) - block_size -1, (batch_size,))
        # x: 입력 시퀀스, y: x를 한칸씩 뒤로 민 시퀀스 (= next-token 정답)
        x = torch.stack([d[i:i+block_size] for i in ix]).to(device)
        y = torch.stack([d[i+1:i+block_size+1] for i in ix]).to(device)

        return x,y

    return get_batch, len(tokenizer) # vocab_size

## 모델 구축 : 트랜스포머 모델 (model.py)

In [4]:
import torch
import torch.nn as nn
import math

In [5]:
#int id -> vector
#d_model : dimension of vector
#vocab_size : how many words in vocabulary
class InputEmbeddings(nn.Module):
    def __init__(self, vocab_size:int, d_model:int):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.d_model)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model:int, block_size:int, dropout: float) -> None:
        super().__init__()
        self.d_model = d_model
        self.block_size = block_size
        self.dropout = nn.Dropout(dropout)

        # Create a matrix of shape (block_size, d_model)
        pe = torch.zeros(block_size, d_model)
         # Create a vector of shape (block_size, 1)
        position = torch.arange(0, block_size, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0,d_model, 2).float()* -(math.log(10000.0)/d_model))
        #Apply the sin to even positions
        pe[:, 0::2] = torch.sin(position * div_term)
        #Apply the cosin to odd positions
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0) # (1, block_size, d_model)
        # keep inside model not as parameter but want to save
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + (self.pe[:, :x.shape[1], :]).requires_grad_(False)
        return self.dropout(x)

class LayerNormalization(nn.Module):
    def __init__(self, eps: float = 1e-6) -> None: # 1e-6 == 10**-6
        super().__init__()
        self.eps = eps
        self.alpha = nn.Parameter(torch.ones(1)) # Multiplied
        self.bias = nn.Parameter(torch.zeros(1)) # Added

    def forward(self, x):
        mean = x.mean(dim = -1, keepdim=True)
        std = x.std(dim = -1, keepdim=True)
        return self.alpha * (x-mean)/ (std+self.eps) + self.bias

class FeedForwardBlock(nn.Module):
    def __init__(self, d_model: int, d_ff:int, dropout: float) -> None:
        super().__init__()
        self.linear_1 = nn.Linear(d_model, d_ff) # W1 and B1
        self.dropout = nn.Dropout(dropout)
        self.linear_2 = nn.Linear(d_ff, d_model) # W2 and B2

    def forward(self, x):
        # (Batch, block_size, d_model) --> (Batch, block_size, d_ff) --> (Batch,block_size, d_model)
        return self.linear_2(self.dropout(torch.relu(self.linear_1(x))))

class MultiHeadAttentionBlock(nn.Module):
    def __init__(self, d_model:int, h: int, dropout:float) -> None:
        super().__init__()
        self.d_model = d_model
        self.h = h
        assert d_model % h == 0, "d_model is not divisible by h"

        self.d_k = d_model // h
        self.w_q = nn.Linear(d_model, d_model) # wq
        self.w_k = nn.Linear(d_model, d_model) # wk
        self.w_v = nn.Linear(d_model, d_model) # wv
        # d_k = d_v
        self.w_o = nn.Linear(d_model, d_model) # wo
        self.dropout = nn.Dropout(dropout)

    @staticmethod
    def attention(query, key, value, mask, dropout: nn.Dropout):
        d_k = query.shape[-1]

        # (Batch, h, block_size, d_k) --> (Batch, h, block_size, block_size)
        attention_scores = (query @ key.transpose(-2,-1)) / math.sqrt(d_k)
        # Before softmax , Apply Mask
        if mask is not None:
            attention_scores.masked_fill_(mask ==0, -1e9)
        attention_scores  = attention_scores.softmax(dim = -1) # (Batch, h, block_size, block_size)
        if dropout is not None:
            attention_scores = dropout(attention_scores)

        return (attention_scores @ value), attention_scores # attention_scores for visualizing
                #(Batch, h, block_size, d_k)

    def forward(self, q, k, v, mask):
        query = self.w_q(q) # (Batch, block_size, d_model) --> (Batch, block_size, d_model)
        key = self.w_k(k)   # (Batch, block_size, d_model) --> (Batch, block_size, d_model)
        value = self.w_v(v) # (Batch, block_size, d_model) --> (Batch, block_size, d_model)

        # Divide to Multi head
        # (Batch, block_size, d_model) --> (Batch, block_size, h, d_k) --> (Batch, h, block_size, d_k)
        # We want Head to watch (block_size, d_k)
        query = query.view(query.shape[0], query.shape[1], self.h, self.d_k).transpose(1,2)
        key = key.view(key.shape[0], key.shape[1], self.h, self.d_k).transpose(1,2)
        value = value.view(value.shape[0], value.shape[1], self.h, self.d_k).transpose(1,2)

        x, self.attention_scores = MultiHeadAttentionBlock.attention(query, key, value, mask, self.dropout)

        # (Batch, h, block_size, d_k) --> (Batch, block_size, h, d_k) --> (Batch, block_size, d_model) ##**(d_model = h*d_k)
        x = x.transpose(1,2).contiguous().view(x.shape[0], -1, self.h * self.d_k)

        # (Batch, block_size,d_model ) --> (Batch, block_size, d_model )
        return self.w_o(x)

class ResidualConnection(nn.Module):
    def __init__(self, dropout: float) -> None:
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.norm = LayerNormalization()

    def forward(self, x, sublayer):
        return x + self.dropout(sublayer(self.norm(x))) # 논문과 다르게 sublayer를 나중에 적용함. 대부분의 코드가 그래서

class EncoderBlock(nn.Module):
    def __init__(self, self_attention_block: MultiHeadAttentionBlock, feed_forward_block : FeedForwardBlock, dropout: float):
        super().__init__()
        self.self_attention_block = self_attention_block
        self.feed_forward_block = feed_forward_block
        self.residual_connections = nn.ModuleList([ResidualConnection(dropout) for _ in range(2)])

    def forward(self, x, src_mask):
        x = self.residual_connections[0](x, lambda x: self.self_attention_block(x,x,x, src_mask))
        x = self.residual_connections[1](x, self.feed_forward_block)
        return x

class Encoder(nn.Module):
    def __init__(self, layers: nn.ModuleList) -> None:
        super().__init__()
        self.layers = layers
        self.norm = LayerNormalization()

    def forward(self, x, mask):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

# 인코더-디코더 구조 모델에서의 디코더 블록 (사용하지 않음)
class DecoderBlock(nn.Module):
    def __init__(self, self_attention_block: MultiHeadAttentionBlock,
                 cross_attention_block:MultiHeadAttentionBlock,
                 feed_forward_block: FeedForwardBlock, dropout: float)->None:
        super().__init__()
        self.self_attention_block = self_attention_block
        self.cross_attention_block = cross_attention_block
        self.feed_forward_block = feed_forward_block
        self.residual_connections = nn.ModuleList([ResidualConnection(dropout) for _ in range(3)])

    def forward(self, x, encoder_output, src_mask, tgt_mask):
        # source language, target language
        x = self.residual_connections[0](x, lambda x: self.self_attention_block(x,x,x, tgt_mask))
        x = self.residual_connections[1](x, lambda x: self.cross_attention_block(x,encoder_output, encoder_output, src_mask))
        x = self.residual_connections[2](x, self.feed_forward_block)
        return x

class Decoder(nn.Module):
    def __init__(self, layers:nn.ModuleList)->None:
        super().__init__()
        self.layers = layers
        self.norm = LayerNormalization()

    def forward(self, x, encoder_output, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)
        return self.norm(x)

# Last layer : project into vocab_size
class ProjectionLayer(nn.Module):
    def __init__(self, d_model:int, vocab_size:int):
        super().__init__()
        self.proj = nn.Linear(d_model, vocab_size)

    def forward(self,x):
        # (Batch, block_size, d_model) --> (Batch, block_size, vocab_size)
        return self.proj(x) # torch.log_softmax(self.proj(x), dim= -1) 수정 logit계산을 위해 log_softmax 제거
                # raw logits 반환

### GPT전용 디코더 블록 (cross-attention 제거)

In [6]:
class GPTDecoderBlock(nn.Module):
    """ 원래 DecoderBlock과 달리 cross-attention 없는 버전.
    self-attention(causal mask) + feed-forward만 사용 """
    def __init__(self,
                 self_attention_block:MultiHeadAttentionBlock,
                 feed_forward_block: FeedForwardBlock, dropout: float) -> None:
        super().__init__()
        self.self_attention_block = self_attention_block
        self.feed_forward_block = feed_forward_block
        self.residual_connections = nn.ModuleList([ResidualConnection(dropout) for _ in range(2)])

    def forward(self, x, tgt_mask):
        x = self.residual_connections[0](x, lambda x: self.self_attention_block(x,x,x, tgt_mask))
        x = self.residual_connections[1](x, self.feed_forward_block)
        return x

class GPTDecoder(nn.Module):
    def __init__(self, layers: nn.ModuleList):
        super().__init__()
        self.layers = layers
        self.norm = LayerNormalization()

    def forward(self, x, tgt_mask):
        for layer in self.layers:
            x = layer(x, tgt_mask)
        return self.norm(x)

### MiniGPT 클래스

In [7]:
class MiniGPT(nn.Module):
    # forward(x,y)를 호출하면 (logits, loss)를 반환하도록
    def __init__(self,
                 decoder: GPTDecoder,
                 tok_embed: InputEmbeddings,
                 pos_embed: PositionalEncoding,
                 projection_layer: ProjectionLayer,
                 block_size:int)-> None:
        super().__init__()
        self.decoder = decoder
        self.tok_embed = tok_embed
        self.pos_embed = pos_embed
        self.projection_layer = projection_layer
        self.block_size = block_size

        #######
        # causal mask buffer에 저장.
        mask = torch.tril(torch.ones(block_size, block_size)).unsqueeze(0).unsqueeze(0)
        self.register_buffer('causal_mask', mask) # (1,1, block_size, block_size)

        # initialize parameter
        # 모델의 모든 가중치(weight)를 처음 학습 시작 전에 어떤 값으로 채워둘지 정하는 부분
        # 없어도 동작하지만
        """ GPT 논문/구현체들은 경험적으로 표준편차 0.02짜리 정규분포로 초기화하면 학습이 안정적으로 잘 된다는 걸 발견해서,
        이게 거의 표준 관례가 됐습니다 (GPT-2 논문에서 유래)
        """
        self.apply(self._init_weights)

    def _init_weights(self, module):
        # GPT류 모델의 초기화 방식: Linear/Embedding: normal분포, LayerNorm계열은 건드리지 않고 그대로)
        if isinstance(module, nn.Linear): # Linear 층이면 가중치는 정규분포, bias = 0
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding): # 임베딩 테이블도 가중치는 정규분포, bias = 0
            nn.init.normal_(module.weight, mean=0.0, std=0.02)



    def forward(self, idx, targets=None): # x: idx, y: targets
        # idx, targets : (Batch, block_size)
        B, T = idx.shape
        x_tok_emb = self.tok_embed(idx) # (Batch, block_size)->(Batch, block_size, d_model)
        x = self.pos_embed(x_tok_emb) #(Batch, block_size, d_model)->(Batch, block_size, d_model)

        tgt_mask = self.causal_mask[:,:,:T,:T] # (1,1,T,T)
        x = self.decoder(x, tgt_mask) # (Batch, block_size, d_model)

        logits = self.projection_layer(x)
        # 주의: ProjectionLayer.forward는 log_softmax까지 적용하지만,
        # cross_entropy는 log_softmax 안 된 raw logits를 받아야 하므로 .proj()만 직접 호출
        # => projection_layer 수정

        loss = None
        if targets is not None:
            loss = nn.functional.cross_entropy(
                logits.view(-1, logits.size(-1)),   #(B*T, vocab_size)
                targets.view(-1)                    #(B*T, )
            )
        return logits, loss
    '''
    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        self.eval()
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.block_size:]) # block_size 넘으면 뒤에서부터 자르기
            probs = torch.softmax(logits[:, -1, :], dim=-1) # 마지막 타임스텝 분포만 사용
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx
    '''
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, stop_ids=None):
        self.eval()

        # stop_ids 추가
        stop_ids = set(stop_ids) if stop_ids else set()

        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.block_size:]) # block_size 넘으면 뒤에서부터 자르기
            probs = torch.softmax(logits[:, -1, :], dim=-1) # 마지막 타임스텝 분포만 사용
            next_id = torch.multinomial(probs, 1)
            idx = torch.cat([idx, next_id], dim=1)

            # batch_size=1 가정: stop 토큰이 나오면 즉시 멈춤
            if stop_ids and next_id.item() in stop_ids:
                break

        return idx

모델 쌓는 함수

In [8]:
def build_gpt(vocab_size: int, block_size:int, d_model:int=384, N:int=6, h:int=6,
              dropout:float=0.1, d_ff:int = None) -> MiniGPT:
    # 디코더 전용 GPT 스타일 모델 생성
    if d_ff is None: # dimension of feed forward
        d_ff = 4 * d_model

    tok_embed = InputEmbeddings(vocab_size, d_model)
    pos_embed = PositionalEncoding(d_model, block_size, dropout)

    decoder_blocks = []
    for _ in range(N):
        self_attention_block = MultiHeadAttentionBlock(d_model,h,dropout)
        feed_forward_block = FeedForwardBlock(d_model, d_ff, dropout)
        decoder_blocks.append(GPTDecoderBlock(self_attention_block, feed_forward_block, dropout))

    decoder = GPTDecoder(nn.ModuleList(decoder_blocks))
    projection_layer = ProjectionLayer(d_model, vocab_size)

    model = MiniGPT(decoder, tok_embed,pos_embed, projection_layer,block_size)
    return model

## 학습 파트 (train.py)

In [9]:
import time, datetime
import torch
import torch.nn as nn

In [20]:
@torch.no_grad()
def estimate_loss(model, get_batch, eval_iters= 20):
  """Average loss over several batches, with dropout off."""
  model.eval()
  # {"train": 평균loss, "val": 평균loss} 형태의 딕셔너리를 한 번에 생성
  out = {
    s: sum(model(*get_batch(s))[1].item() for _ in range(eval_iters)) / eval_iters
        # get_batch(s) -> (x, y) 튜플 -> model(*get_batch(s)) == model(x, y)
        # model(x, y) -> (logits, loss) 튜플 -> [1] 로 loss만 추출 -> .item()으로 python 숫자로 변환
        # 이 과정을 eval_iters 20번 반복해서 평균 -> 배치 하나만 보고 판단하면 노이즈가 크므로 여러 번 측정해서 평균냄
         for s in ("train", "val")
      }
  model.train()
  return out

1. from scratch train.py

In [21]:
def train(model, get_batch, device, lr=3e-4, steps=5000, eval_interval=100,
          decode=None, save_path="mini_gpt.pt"):
  """
  model          : build_gpt()로 만든 Transformer
  get_batch      : split('train'/'val')을 인자로 받아 (x, y) 텐서를 반환하는 함수
  device         : 'cuda' 또는 'cpu'
  lr             : 학습률
  steps          : 전체 학습 스텝 수
  eval_interval  : 몇 step마다 train/val loss를 측정해서 출력할지
  decode         : 토큰 id 리스트 -> 문자열 (샘플 출력용, 없으면 토큰 id를 그대로 출력)
  save_path      : 학습된 가중치를 저장할 파일 경로
  """
  model.to(device)
  model.train()
  # 모델 파라미터 총 개수를 천 단위 구분기호와 함께 출력
  print(f"{sum(p.numel() for p in model.parameters()):,} parameters, device={device}")

  opt = torch.optim.AdamW(model.parameters(), lr=lr)

  start = time.time()
  for step in range(steps):
    x, y = get_batch("train")  # x,y : (batch_size, block_size)
    _, loss = model(x,y)  # forward
    opt.zero_grad()
    loss.backward()
    opt.step()

    if step % eval_interval == 0 or step == steps -1:
      est = estimate_loss(model, get_batch) # train/val 평균 loss 측정
      sec = time.time()-start
      strtime = str(datetime.timedelta(seconds=sec)).split(".")[0]
      print(f"step {step:5d}/{steps} train {est['train']:.3f} val {est['val']:.3f} time {strtime}", flush=True)

  torch.save(model.state_dict(), save_path)
  print("saved weights to {save_path}")

  # 학습 끝난 후 평가 모드. 텍스트 생성 데모
  model.eval()
  prompt = torch.zeros((1,1), dtype=torch.long, device=device) # start token (token id 0)
  print("\n--- sample ---")
  # generate 500 tokens
  generated = model.generate(prompt, 500)[0].tolist()
  # decode 함수가 주어졌을 경우, 사람이 읽을 텍스트로 변환.
  print(decode(generated) if decode is not None else generated)

디코드 함수 (tokenizer의 decode 사용)

In [10]:
def decode(token_ids):
  return tokenizer.decode(token_ids)

하이퍼 파라미터 설정

In [11]:
block_size = 64
batch_size = 32

d_model = 256
N = 4
h = 4
dropout = 0.1

데이터 로드 & 모델 생성

In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load data by batch size
get_batch, vocab_size = load_chatbot_data(block_size=block_size, batch_size=batch_size, device=device)



Generating train split: 0 examples [00:00, ? examples/s]

1. From Scratch model (Transformer-decoder)

In [22]:
# Create model(from scratch)
model = build_gpt(vocab_size=vocab_size, block_size=block_size,
                      d_model=d_model, N=N, h=h, dropout=dropout)

학습

In [23]:
# TRAIN
train(model, get_batch, device, lr=3e-4, steps=4000, eval_interval=100, decode=decode)


29,421,075 parameters, device=cuda
step     0/4000 train 10.581 val 10.618 time 0:00:01
step   100/4000 train 5.859 val 6.184 time 0:00:11
step   200/4000 train 4.899 val 5.317 time 0:00:20
step   300/4000 train 4.346 val 4.817 time 0:00:29
step   400/4000 train 4.025 val 4.628 time 0:00:38
step   500/4000 train 3.723 val 4.515 time 0:00:48
step   600/4000 train 3.496 val 4.415 time 0:00:57
step   700/4000 train 3.275 val 4.315 time 0:01:07
step   800/4000 train 3.118 val 4.211 time 0:01:17
step   900/4000 train 2.964 val 4.156 time 0:01:27
step  1000/4000 train 2.774 val 4.207 time 0:01:38
step  1100/4000 train 2.653 val 4.166 time 0:01:48
step  1200/4000 train 2.496 val 4.155 time 0:01:58
step  1300/4000 train 2.340 val 4.179 time 0:02:09
step  1400/4000 train 2.217 val 4.176 time 0:02:20
step  1500/4000 train 2.103 val 4.242 time 0:02:31
step  1600/4000 train 1.961 val 4.189 time 0:02:41
step  1700/4000 train 1.831 val 4.271 time 0:02:52
step  1800/4000 train 1.724 val 4.301 time 0:

## 채팅 (chat.py)

In [24]:
def chat(model, tokenizer, checkpoint_path="mini_gpt.pt",
         device="cpu", max_new_tokens=100, stop_tokens=("<usr>",),
):
    """
    학습된 GPT 모델을 로드해서 터미널 기반 챗봇 인터페이스를 실행

    Args:
        model: block_size, generate(idx, max_new_tokens, stop_ids) 를 갖춘 모델 객체 (가중치 로드 전 상태)
        tokenizer: encode/decode, eos_token_id, convert_tokens_to_ids 를 지원하는 토크나이저
        checkpoint_path: 학습된 state_dict 파일 경로
        device: "cpu" 또는 "cuda"
        max_new_tokens: 답변 생성 시 최대 토큰 수
        stop_tokens: eos 외에 생성을 멈출 추가 토큰들 (기본: "<usr>")
    """
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.to(device)
    model.eval()

    eos_id = tokenizer.eos_token_id
    stop_ids = [eos_id] + [tokenizer.convert_tokens_to_ids(t) for t in stop_tokens]

    print("Type a prompt and the model will continue it (empty line or Ctrl-D to quit)")
    while True:
          try:
               prompt = input("> ")
          except EOFError:
               break
          if not prompt:
               break

          formatted = f"<usr> {prompt} <bot>"
          ids = tokenizer.encode(formatted)
          if not ids:
               print("(no tokens from the prompt are in the vocabulary)")
               continue

          if len(ids) >= model.block_size:
               ids =  ids[-(model.block_size -1):]

          idx = torch.tensor([ids], dtype=torch.long, device=device)
          out = model.generate(idx, max_new_tokens, stop_ids=stop_ids)[0].tolist()

          generated_ids = out[len(ids):]
          if generated_ids and generated_ids[-1] in stop_ids:
               generated_ids = generated_ids[:-1]

          response = tokenizer.decode(generated_ids, skip_special_tokens=True)
          print(response)


학습할 때 사용한 모델과 동일한 모델 생성

In [25]:
model = build_gpt(vocab_size=vocab_size, block_size=block_size,
                      d_model=d_model, N=N, h=h, dropout=dropout)

채팅 테스트

In [27]:
chat(model, tokenizer, checkpoint_path="mini_gpt.pt", device="cpu")

Type a prompt and the model will continue it (empty line or Ctrl-D to quit)
> 오늘은 날씨가 어때요
부모님만의 사정이 있을거예요. 이해하려고 노력해보세요.

> 안녕하세요.
안녕하세요.

> 놀자.
같이 놀아요.

> 
